In [ ]:
import os, glob
from natsort import natsorted

import cytoflow as flow
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd
from pathlib import Path

import ipywidgets as widgets
from IPython.display import display

from flow_analysis_tools import flow_violin
from utils import *
from utils.color_palettes import darken_palette
from utils import CONSTRUCT_NAME_DICT, GUIDE_NAME_DICT
from utils.io import n_underscore_constructor

from rclone_tools import NodeExperimentsDirectory

In [ ]:
DRIVE_RELPATH = "ARS_Working/Experiments/4_phHSC_Testing/260914_phHSCc12R1_Test/261006_phHSCc21R1_timecourse_analysis"
DATA_DIR = "/data_gilbert3/asathler/hsiung-box"
PROXY = "http://169.230.134.132:3128"

In [ ]:
Experiment = NodeExperimentsDirectory(
    drive="hsiung",
    source_relpath=DRIVE_RELPATH,
    node_mountpath=DATA_DIR,
    output_dirname="out",
    proxy=PROXY
)

In [ ]:
Experiment.stage()

In [ ]:
%matplotlib widget
matplotlib.rc('figure', dpi = 160)
plt.style.use('ggplot')

In [ ]:
FCS_FILES = sorted([Path(x) for x in glob.glob(os.path.join(Experiment.dest_dirpath, "*.fcs"))])
FCS_NAMES = [file.stem for file in FCS_FILES]
DATE_OFFSET = 3
YLIMITS = (0, 7)
CHANNELS = {
    "VL1-A": ["Cas", "mTagBFP2"],
    "BL1-A": ["Guide", "eGFP"],
    "RL1-A": ["CD55", "APC"],
    "YL1-A": ["CD81", "PE"]
}
CHANNEL_CMAPS = {
    "VL1-A": black_blue_pastel_palette,
    "BL1-A": black_green_pastel_palette,
    "RL1-A": black_red_pastel_palette,
    "YL1-A": black_yellow_pastel_palette,
}
ALL_NEG_CONSTRUCT = "phHSC"
GATE_INFO = {}

CONSTRUCTOR_MAP = [
    ("Date", 0, CONSTRUCT_NAME_DICT, "category"),
    ("Construct", -3, CONSTRUCT_NAME_DICT, "category"),
    ("Treatment", -2, GUIDE_NAME_DICT, "category"),
    ("Guide", -1, GUIDE_NAME_DICT, "category")
]
NAME_FUNCTIONS_DICT = {name: n_underscore_constructor(idx, this_dict) for name, idx, this_dict, _ in CONSTRUCTOR_MAP}

CHANNELS_OVERRIDE = {
    "VL1-A": "VL1-A"
}

print('\n'.join([f"'{x}'" for x in FCS_FILES]))
if isinstance(Experiment, NodeExperimentsDirectory):
    for x in FCS_FILES: Experiment.logger.info(x)

In [ ]:
max_fcs_len = max([len(x) for x in FCS_NAMES]) + 5
name_categories = {"names": FCS_NAMES}
for cat_name, cat_func in NAME_FUNCTIONS_DICT.items():
    name_categories[cat_name] = [cat_func(this_name) for this_name in FCS_NAMES]
name_cat_df = pd.DataFrame(name_categories)
print(name_cat_df.head(n=len(name_cat_df)))

import_op = flow.ImportOp(
    conditions = {
        name: this_type for name, _, _, this_type in CONSTRUCTOR_MAP
    },
    tubes = [flow.Tube(file=fcs_file, conditions={
        name: this_function(fcs_name) for name, this_function in NAME_FUNCTIONS_DICT.items()
    }) for fcs_file, fcs_name in zip(FCS_FILES, FCS_NAMES)],
    channels = CHANNELS_OVERRIDE
)

try:
    ex = import_op.apply()
except ValueError as e:
    error_text = str(e)
    if error_text == "Categorical categories cannot be null":
        raise Exception(
            "Functions 'get_construct_name' or 'get_guide_type' likely are turning None or NoneType." + \
            "\n\t   This is typically because a construct or guide name case isn't properly handled."
        )
    else:
        raise e

# Gating Single Cells

In [ ]:
single_gate = flow.PolygonOp(
    name="Single_Cell",
    xchannel="FSC-A",
    ychannel="SSC-A"
)

single_gate.default_view(
    density = True,
    huescale = "log",
    interactive = True
).plot(ex, gridsize = 100)

In [ ]:
if isinstance(Experiment, NodeExperimentsDirectory):
    plt.savefig(Experiment.output_dirpath / f"single_cell_gate.png")

In [ ]:
GATE_INFO["single_cell_vertices"] = np.asarray(single_gate.vertices)
ex = single_gate.apply(ex)
plt.close()

# Single Gate Ops

In [ ]:
plt.ioff()

channels = list(CHANNELS) if len(CHANNELS_OVERRIDE) == 0 else list(CHANNELS_OVERRIDE.keys())
# subset = f"(Single_Cell == True) and (Construct == '{ALL_NEG_CONSTRUCT}')"
subset= f"Construct == '{ALL_NEG_CONSTRUCT}'"
gates = {}

out = widgets.Output()
status = widgets.Label()
btn = widgets.Button(description="Accept threshold", button_style="success")

# Keep strong references: matplotlib holds event callbacks weakly, so if the
# view is garbage-collected, its click handler silently disappears.
state = {"i": 0, "op": None, "view": None, "fig": None}

def show_channel(i):
    chan = channels[i]
    op = flow.ThresholdOp(name=chan, channel=chan)
    view = op.default_view(huescale="log", interactive=True,
                           subset=subset, scale="log")
    plt.close("all")
    with out:
        out.clear_output(wait=True)
        view.plot(ex)
        fig = plt.gcf()
        display(fig.canvas)

    state.update(op=op, view=view, fig=fig)
    status.value = f"[{i+1}/{len(channels)}] {chan}: click the plot, then Accept"

def on_accept(_b):
    op = state["op"]
    if op.threshold is None:
        status.value = "No threshold set yet; click the plot first"
        return
    gates[op.channel] = op
    print(f"{op.channel}: {op.threshold}")
    state["i"] += 1
    if state["i"] < len(channels):
        show_channel(state["i"])
    else:
        plt.close(state["fig"])
        btn.disabled = True
        status.value = f"Done: {len(gates)} gates stored in `gates`"

btn.on_click(on_accept)
display(widgets.VBox([out, widgets.HBox([btn, status])]))
show_channel(0)
print(len(state["fig"].axes))

In [ ]:
for chan, thresh_op in gates.items():
    GATE_INFO[f"{chan}_threshold"] = thresh_op.threshold
    for char in ["-", "_"]:
        thresh_op.name = thresh_op.name.replace(char, "")
    ex = thresh_op.apply(ex)

In [ ]:
plt.ion()

# Quad Op

In [ ]:
quad_gate = flow.QuadOp(
    name="d_pos",
    xchannel="VL1-A",
    ychannel="BL1-A",
    # xscale="logicle",
    # yscale="logicle"
)

quad_gate.default_view(
    density = True,
    huescale = "log",
    interactive = True,
    subset = f"(Single_Cell == True) & (Construct == '{ALL_NEG_CONSTRUCT}')",
    xscale="log", yscale="log"
).plot(ex, gridsize=100)

In [ ]:
plt.savefig(Experiment.output_dirpath / "quad_gate.png")
GATE_INFO["quad_gate"] = {
    "x": quad_gate.xthreshold,
    "y": quad_gate.ythreshold
}

In [ ]:
ex = quad_gate.apply(ex)
plt.close()

# Timeseries
Based on secondary

In [ ]:
%matplotlib inline

In [ ]:
# plot_data = ex.data[ex.data["Single_Cell"] == True]
PLOT_CHANNEL = "VL1-A"
PALETTE_MAP = {
    "pARS001": black_blue_pastel_palette,
    "pCH45": black_green_pastel_palette,
    "pRA02": black_red_pastel_palette
}

In [ ]:
ex.data['Construct'].unique()

In [ ]:
ex.data['Date'] = pd.to_datetime(
    ex.data['Date'],
    format="%y%m%d"
)

In [ ]:
ex.data['Date'].unique()

In [ ]:
guide_vals = ['5', '20', '60', '155']

In [ ]:
fig, axs = plt.subplot_mosaic(
    np.array(guide_vals).reshape((2,2)),
    figsize=(12, 12),
    height_ratios=(1,1),
    width_ratios=(1,1)
)

for g_val in guide_vals:
    
    flow_violin(
        data=ex.data[(ex.data["Guide"] == g_val)],
        x="Date",
        y=PLOT_CHANNEL,
        hue="Construct",
        hue_order=['dCas9-ZIM3', 'multiAsCas12a-KOX1'],
        log_y_axis=True,
        ax=axs[g_val],
        density_norm="width"
    )

    g_val_int = int(g_val)
    axs[g_val].set_xticklabels(["Day 3 (p3)", "Day 8 (p4)", "Day 10 (p5)"])
    axs[g_val].set_title(f"{g_val_int}uL ({round((g_val_int / (1000+g_val_int)) * 100, 1)}% Supernatant)")

plt.subplots_adjust()
plt.savefig(Experiment.output_dirpath / "timeseries.png")
plt.savefig(Experiment.output_dirpath / "timeseries.svg")

In [ ]:
ex.data.to_csv(Experiment.output_dirpath / f"processed_data.csv")
ex.data.to_csv(Experiment.output_dirpath / f"groupby_means.csv")

In [ ]:
np.savez(
    file=Experiment.output_dirpath / "gate_info.npz",
    **GATE_INFO
)

In [ ]:
Experiment.close()